# Glottis Demo

Generate audio from the Pink Trombone glottis model with animated parameters.

Note: this is the raw glottis waveform (no vocal tract filtering yet).
It sounds buzzy/electronic because the tract is what shapes it into vowels.


In [ ]:
import torch
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
    SAMPLES_PER_FRAME,
)

In [ ]:
def make_params(duration_s=2.0, **overrides):
    """Create a parameter tensor with defaults, allowing per-parameter overrides.

    Each override can be a scalar (constant) or a 1D tensor (one value per control frame).
    """
    T = int(duration_s * 12.5)  # control rate = 12.5 Hz
    defaults = {
        "noise": 0.1,
        "frequency": 140,
        "tenseness": 0.6,
        "intensity": 1,
        "loudness": 1,
        "tongueIndex": 12.9,
        "tongueDiameter": 2.43,
        "vibratoWobble": 0,
        "vibratoFrequency": 6,
        "vibratoGain": 0.0,
        "tractLength": 44,
    }
    params = torch.zeros(1, T, N_PARAMS)
    for i, name in enumerate(PARAM_NAMES):
        val = overrides.get(name, defaults.get(name, 0))
        if isinstance(val, torch.Tensor):
            params[0, :, i] = val[:T]
        else:
            params[0, :, i] = val
    return params


def play(params, normalize=True):
    """Synthesize and play audio from a parameter tensor."""
    with torch.no_grad():
        audio = pink_trombone(params)
    wav = audio[0]
    if normalize and wav.abs().max() > 0:
        wav = wav / wav.abs().max() * 0.9
    display(Audio(wav, rate=SAMPLE_RATE))
    return wav

## 1. Steady vowel

Constant pitch at 140 Hz, no vibrato.


In [ ]:
params = make_params(duration_s=2.0)
_ = play(params)

## 2. Pitch sweep (100 Hz -> 300 Hz)


In [ ]:
T = 50  # 4 seconds
freq_sweep = torch.linspace(100, 300, T)
params = make_params(duration_s=4.0, frequency=freq_sweep)
_ = play(params)

## 3. Tenseness sweep (breathy -> tense)

Low tenseness = breathy/soft, high tenseness = crisp/sharp.


In [ ]:
T = 50
tens_sweep = torch.linspace(0.1, 0.95, T)
params = make_params(duration_s=4.0, tenseness=tens_sweep)
_ = play(params)

## 4. Vibrato variations


In [ ]:
# Singing-style vibrato
print("Singing vibrato (5.5 Hz, moderate depth):")
params = make_params(
    duration_s=3.0, frequency=220, vibratoGain=0.04, vibratoFrequency=5.5
)
_ = play(params)

# With wobble (natural pitch drift)
print("With wobble (organic pitch wandering):")
params = make_params(
    duration_s=3.0,
    frequency=220,
    vibratoGain=0.02,
    vibratoFrequency=5.5,
    vibratoWobble=1.0,
)
_ = play(params)

## 5. Intensity fade in/out


In [ ]:
T = 50
intensity = torch.cat(
    [
        torch.linspace(0, 1, T // 2),
        torch.linspace(1, 0, T - T // 2),
    ]
)
params = make_params(duration_s=4.0, intensity=intensity, frequency=180)
_ = play(params)

## 6. Aspiration noise

The `noise` parameter adds breathiness by injecting a periodic buzz modulated by the
glottal cycle. It's most audible at low tenseness. (True frication/hissing
comes from the tract constrictions, not yet implemented.)


In [ ]:
# Breathy voice: low tenseness + high noise
print("Breathy (tenseness=0.2, noise=1):")
params = make_params(duration_s=3.0, noise=1.0, frequency=140, tenseness=0.2)
_ = play(params)

# Clean voice for comparison
print("Clean (tenseness=0.6, noise=0):")
params = make_params(duration_s=3.0, noise=0.0, frequency=140, tenseness=0.6)
_ = play(params)

## 7. Musical phrase

Simple melody with intensity envelope.


In [ ]:
# C4-E4-G4-C5 arpeggio, each note ~0.5s
notes_hz = [261.6, 329.6, 392.0, 523.3]
frames_per_note = 6  # 0.48s per note
T = frames_per_note * len(notes_hz)

freq = torch.zeros(T)
intensity = torch.zeros(T)
for i, f in enumerate(notes_hz):
    start = i * frames_per_note
    freq[start : start + frames_per_note] = f
    # Attack-sustain-release envelope per note
    intensity[start] = 0.3
    intensity[start + 1 : start + frames_per_note - 1] = 1.0
    intensity[start + frames_per_note - 1] = 0.3

params = make_params(
    duration_s=T / 12.5,
    frequency=freq,
    intensity=intensity,
    tenseness=0.7,
    vibratoGain=0.01,
    vibratoFrequency=5.0,
)
_ = play(params)

## 8. Gradient check

Verify that gradients flow through the synthesizer.


In [ ]:
params = make_params(duration_s=1.0)
params.requires_grad_(True)

audio = pink_trombone(params)
loss = audio.pow(2).mean()
loss.backward()

grad = params.grad[0, 0]  # gradients for first frame
print("Gradients per parameter:")
for i, name in enumerate(PARAM_NAMES):
    g = grad[i].item()
    print(f"  {name:>25s}: {g:+.6f}")